In [ ]:
# !!! These lines should be executed in the bash directly !!!
# Initialize environment
! graphrag init
# Starting the indexing
! graphrag index

In [10]:
! graphrag query "What are the top themes in this story?"

# Top Themes in *A Christmas Carol*  

The story of *A Christmas Carol* by Charles Dickens is a rich tapestry of interconnected themes, each explored through the transformation of Ebenezer Scrooge and the broader social commentary embedded in the narrative. The following sections synthesize the key themes identified by analysts, supported by data references from the dataset.  

---

## 1. **Redemption and Moral Transformation**  
The central theme of the story revolves around Scrooge’s journey from a miserly, self-centered individual to a compassionate and generous person. This transformation is driven by supernatural encounters with the Ghosts of Christmas Past, Present, and Future, which expose the consequences of his greed and isolation. Analysts emphasize that Scrooge’s redemption is not merely a personal awakening but a moral reckoning with his past actions, including his neglect of a child at his school and his harsh treatment of Bob Cratchit. The narrative underscores the power 

In [9]:
! graphrag query \"Who is Scrooge and what are his main relationships?"\ --method local

Usage: graphrag query [OPTIONS] QUERY
Try 'graphrag query --help' for help.
┌─ Error ─────────────────────────────────────────────────────────────────────┐
│ Got unexpected extra arguments (is Scrooge and what are his main            │
│ relationships?\ --method local)                                             │
└─────────────────────────────────────────────────────────────────────────────┘


In [1]:
import os
from pathlib import Path
from pprint import pprint

import graphrag
import graphrag.api as api
import pandas as pd
from graphrag.config.load_config import load_config
from graphrag.index.typing.pipeline_run_result import PipelineRunResult

In [2]:
graphrag_config = load_config(os.getenv("PROJECT_DIRECTORY"))
PROJECT_DIRECTORY = os.getenv("PROJECT_DIRECTORY")

In [11]:
index_result : list[PipelineRunResult] = await api.build_index(
    config=graphrag_config,
    )
# index_result is a list of workflows that make up the indexing pipeline that was run
for workflow_result in index_result:
    status = f"error\n{workflow_result.error}" if workflow_result.error else "success"
    print(f"Workflow Name: {workflow_result.workflow}\tStatus: {status}")

c:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv\Lib\site-packages\graphrag\index\operations\extract_graph\extract_graph.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_relationships = pd.concat(relationship_dfs, ignore_index=False)


Workflow Name: load_input_documents	Status: success
Workflow Name: create_base_text_units	Status: success
Workflow Name: create_final_documents	Status: success
Workflow Name: extract_graph	Status: success
Workflow Name: finalize_graph	Status: success
Workflow Name: extract_covariates	Status: success
Workflow Name: create_communities	Status: success
Workflow Name: create_final_text_units	Status: success
Workflow Name: create_community_reports	Status: success
Workflow Name: generate_text_embeddings	Status: success


In [3]:
entities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/entities.parquet")
communities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/communities.parquet")
text_unit_df = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/text_units.parquet")
relationships = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/relationships.parquet")
#text_units = graphrag.query.indexer_adapters.read_indexer_text_units(text_unit_df)
community_reports = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/community_reports.parquet")

response, context = await api.local_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units= text_unit_df,
    relationships= relationships,
    covariates = None,
    community_level=2,
    response_type="Kurze präzise Antwort auf Deutsch",
    query="Wie heißt dieser Prozess?"
)

In [4]:
response

'Der Prozess wird im BPMN-Diagramm als **„Dienstreiseplanung und Reisekostenabrechnung“** bezeichnet. Er umfasst Schritte wie die Planung der Dienstreise, die Prüfung des Antrags durch die Personalabteilung, die Erstellung der Reisekostenabrechnung sowie die Weiterleitung an die Buchhaltung und Rechnungsverwaltung [Daten: Sources (0), Entities (3, 6, 17), Relationships (3, 6, 15)].'

In [5]:
context

{'reports':   id                                title  \
 0  0  Buchhaltung und Rechnungsverwaltung   
 
                                              content  
 0  # Buchhaltung und Rechnungsverwaltung\n\nDie G...  ,
 'relationships':     id                           source                           target  \
 0    3          DIENSTREISE IST GEPLANT                      BUCHHALTUNG   
 1    6          LANDESREISEKOSTENGESETZ                DIENSTREISEANTRAG   
 2    8                DIENSTREISEANTRAG                      BUCHHALTUNG   
 3    9          DIENSTREISE IST GEPLANT          LANDESREISEKOSTENGESETZ   
 4   19          LANDESREISEKOSTENGESETZ  REISEKOSTENABRECHNUNG_ERSTELLEN   
 5   20          LANDESREISEKOSTENGESETZ         DIENSTREISEANTRAG_SENDEN   
 6   18          LANDESREISEKOSTENGESETZ             DIENSTREISEANTRETTEN   
 7   21                      BUCHHALTUNG              RECHNUNGSVERWALTUNG   
 8   23                      BUCHHALTUNG                             END

In [7]:
queries = ["Wie heißt dieser Prozess?","Wenn ich mich gerade vor einer Dienstreise befinde, was gilt dann für mich zu beachten?",
           "Welcher Schritt kommt direkt nach 'Dienstreise antreten' und wer führt ihn aus?", "Welche Dokumente spielen im laufe des Prozesses eine Rolle und für wen?"]
responses = []
for i, query in enumerate(queries):
    response, context = await api.local_search(
        config=graphrag_config,
        entities=entities,
        communities=communities,
        community_reports=community_reports,
        text_units= text_unit_df,
        relationships= relationships,
        covariates = None,
        community_level=2,
        response_type="Kurze präzise Antwort auf Deutsch",
        query=query
    )
    responses.append(response)
responses

['Der Prozess wird im Kontext der Daten nicht explizit benannt, ist jedoch eng mit der **Dienstreiseplanung** und der **Finanzverwaltung** verbunden. Er umfasst Schritte wie die Beantragung einer Dienstreise, Prüfung durch die Personalabteilung, Reisekostenabrechnung und Buchhaltung. \n\n[Daten: Entities (3, 6, 17), Relationships (3, 8, 21)]',
 'Wenn Sie sich vor einer Dienstreise befinden, sind folgende Punkte zu beachten:  \n1. **Dienstreiseantrag erstellen**: Füllen Sie den Antrag aus und lassen Sie ihn von Vertreter:in und Vorgesetzten unterschreiben [Daten: DIENSTREISEANTRAG (6), VORGESETZTEN (4)].  \n2. **Kostenregelungen prüfen**: Stellen Sie sicher, dass die Reise im Einklang mit dem **Landesreisekostengesetz** (5) erfolgt, da dieses die Kostenregelungen festlegt [Daten: LANDESREISEKOSTENGESETZ (5, 6, 9, 18)].  \n3. **Dokumente bereitstellen**: Senden Sie den Antrag sowie Rechnungen und Belege an die **Personalabteilung** (1), die die Prüfung übernimmt [Daten: DIENSTREISEANTRET